# Paper A1 figures

The eight A1 figures, drawn from what `experiments/a1/run_a1.py` saved (`yg_eo_soilnet.a1.figures`). Each is shown here and saved as PNG and PDF in the run folder's `figures/`.

Use the **explore** kernel (`pixi run -e explore notebook`). Make the outputs first:

```
python experiments/a1/make_plans.py --config configs/a1/a1_demo.yml
python experiments/a1/run_a1.py --config configs/a1/a1_demo.yml --groups
```

A figure whose plans the run does not hold says which ones to run.

In [ ]:
import os
import sys
from pathlib import Path

# GDAL needs PROJ's database, which a kernel started straight from the env cannot find; set before geopandas.
os.environ.setdefault("PROJ_DATA", str(Path(sys.prefix) / "share" / "proj"))

# Work from the repository root, where the settings' relative paths start.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

SETTINGS = "configs/a1/a1_demo.yml"  # configs/a1/a1.yml for the real data
RUN_NAME = "default"
# The demo's regions come from a column; this file holds their outlines. None for the real data, whose
# regions are the settings' own polygons (the sub-basins).
REGIONS = "experiments/a1/demo_data/regions.geojson"
PLAN = "random"  # the plan figures 3 and 8 read
RELIEF = True  # shaded relief under the map (downloaded once, then cached)
N_BOOT = 1000  # bootstrap draws for figures 2 and 4

In [ ]:
import matplotlib.pyplot as plt

from yg_eo_soilnet.a1 import figures
from yg_eo_soilnet.a1.config import load_a1_config
from yg_eo_soilnet.a1.figures import (
    figure_class_scores,
    figure_correlations,
    figure_dissimilarity,
    figure_mixed,
    figure_points_map,
    figure_shap,
    figure_stage1,
    figure_transfer,
)

a1 = load_a1_config(SETTINGS)
inputs = figures.FigureInputs.from_run(
    a1, RUN_NAME, regions_path=REGIONS, regions_name_field="name" if REGIONS else None
)
FOLDER = inputs.run_dir / "figures"
print(f"Reading {inputs.run_dir}; saving to {FOLDER}")

**Figure 1.** The points by measured carbonate class, the region outlines, and how many points of each class every region holds.

In [ ]:
figure = figure_points_map(inputs, relief=RELIEF)
figures.save_figure(figure, "fig1_points_map", FOLDER)
plt.close(figure)  # shown once, as this cell's result, and not again by the inline backend
figure

**Figure 2.** (a) Spearman ρ between SOM and each covariate within each carbonate class; a dot marks a correlation the spatial block bootstrap tells apart from 0 (Holm-corrected over the whole grid). (b) The median bare-soil Sentinel-2 spectrum of each class.

In [ ]:
figure = figure_correlations(inputs, n=N_BOOT)
figures.save_figure(figure, "fig2_correlations", FOLDER)
plt.close(figure)  # shown once, as this cell's result, and not again by the inline backend
figure

**Figure 3.** Stage 1: measured against out-of-fold predicted CaCO₃ and CEC on the plan's training points, and how often the predicted carbonate class matches the measured one.

In [ ]:
figure = figure_stage1(inputs, plan_id=PLAN)
figures.save_figure(figure, "fig3_stage1", FOLDER)
plt.close(figure)  # shown once, as this cell's result, and not again by the inline backend
figure

**Figure 4.** RPIQ and bias per carbonate class for the four stage-2 models, random split against spatial block cross-validation, with 95 % block-bootstrap intervals. Hollow markers: fewer than 30 test points.

In [ ]:
figure = figure_class_scores(inputs, n=N_BOOT)
figures.save_figure(figure, "fig4_class_scores", FOLDER)
plt.close(figure)  # shown once, as this cell's result, and not again by the inline backend
figure

**Figure 5.** (a) Each held-out region's RMSE when trained on the other regions, against its local RMSE. (b) The transfer drop per covariate group.

In [ ]:
figure = figure_transfer(inputs)
figures.save_figure(figure, "fig5_transfer", FOLDER)
plt.close(figure)  # shown once, as this cell's result, and not again by the inline backend
figure

**Figure 6.** How the test RMSE falls as a share of the target region joins training: mixed (other regions + the share) against control (the share alone), one panel per covariate group.

In [ ]:
figure = figure_mixed(inputs)
figures.save_figure(figure, "fig6_mixed", FOLDER)
plt.close(figure)  # shown once, as this cell's result, and not again by the inline backend
figure

**Figure 7.** The transfer drop against how different the region is from its training data: the median dissimilarity index, and the change in the SOM–CEC correlation; colour is the difference in carbonate mix.

In [ ]:
figure = figure_dissimilarity(inputs)
figures.save_figure(figure, "fig7_dissimilarity", FOLDER)
plt.close(figure)  # shown once, as this cell's result, and not again by the inline backend
figure

**Figure 8.** What `global_aux` relies on most within each carbonate class: mean |SHAP| over the plan's test points (the model is refitted exactly as the run fitted it).

In [ ]:
figure = figure_shap(inputs, plan_id=PLAN)
figures.save_figure(figure, "fig8_shap", FOLDER)
plt.close(figure)  # shown once, as this cell's result, and not again by the inline backend
figure

All eight at once, without showing them (for a script or a rerun):

In [ ]:
# figures.render_all(inputs, FOLDER, relief=RELIEF, n_boot=N_BOOT, shap_plan=PLAN, close=True)